# Model evaluation: generate data and exploratory figures
Generates plot-level and grid-level arrays for both paper figures, saves them to disk, and makes exploratory spatial hexbin figures.

In [ ]:
import os

import joblib
import numpy as np
import shap
import xarray as xr

from western_us_biomass.train_models import (
    train_all_models,
    train_model_delta_burned,
    train_model_delta_unburned,
    train_model_init_biomass,
    train_models_utils,
)

In [2]:
OUT_DIR = "../figure_data/figure_S_SHAP/"
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
dir_models = "/dfs10/jranders_lab1/users/czarakas/uncertain_land_sink_data/processed_data/models/"

fia_data = train_all_models.load_data()
fia_data = fia_data.where(
    fia_data["STATECD"].isin([4, 6, 8, 16, 30, 32, 35, 41, 49, 53, 56]).load(), drop=True
)

## Collect plot-level predictions across model folds

In [4]:
ensemble_list = [f"_{i:04d}" for i in range(100)]

In [5]:
y_test_init_actual_list = []
y_test_burned_actual_list = []
y_test_undisturbed_actual_list = []
y_test_init_pred_list = []
y_test_burned_pred_list = []
y_test_undisturbed_pred_list = []

plotid_init_list = []
plotid_burned_list = []
plotid_undisturbed_list = []

lat_init_list = []
lat_burned_list = []
lat_undisturbed_list = []
lon_init_list = []
lon_burned_list = []
lon_undisturbed_list = []

for model_suffix in ensemble_list[0:1]:

    fname = dir_models + "train_plotids" + model_suffix + ".nc"
    print(fname)
    test_train_split = xr.open_dataset(fname, engine="netcdf4")
    model_initial_biomass = joblib.load(
        train_model_init_biomass.FPATH_MODEL + model_suffix + ".pkl"
    )
    model_burned = joblib.load(train_model_delta_burned.FPATH_MODEL + model_suffix + ".pkl")
    model_undisturbed = joblib.load(train_model_delta_unburned.FPATH_MODEL + model_suffix + ".pkl")

    fia_data_test = fia_data.where(~test_train_split["is_train"], drop=True)
    fia_data_test_burned, fia_data_test_undisturbed = train_all_models.split_subcomponents(
        fia_data_test
    )

    [X_test_init, y_test_init] = train_models_utils.get_X_y(
        fia_data=fia_data_test,
        fia_variable_predictors=train_model_init_biomass.FIA_VARIABLE_PREDICTORS,
        predictors_meas1=train_model_init_biomass.PREDICTOR_VARIABLES_MEAS1,
        predictors_meas2=train_model_init_biomass.PREDICTOR_VARIABLES_MEAS2,
        output_variable=train_model_init_biomass.OUTPUT_VARIABLE,
    )
    [X_test_burned, y_test_burned] = train_models_utils.get_X_y(
        fia_data=fia_data_test_burned,
        fia_variable_predictors=train_model_delta_burned.FIA_VARIABLE_PREDICTORS,
        predictors_meas1=train_model_delta_burned.PREDICTOR_VARIABLES_MEAS1,
        predictors_meas2=train_model_delta_burned.PREDICTOR_VARIABLES_MEAS2,
        output_variable=train_model_delta_burned.OUTPUT_VARIABLE,
    )
    [X_test_undisturbed, y_test_undisturbed] = train_models_utils.get_X_y(
        fia_data=fia_data_test_undisturbed,
        fia_variable_predictors=train_model_delta_unburned.FIA_VARIABLE_PREDICTORS,
        predictors_meas1=train_model_delta_unburned.PREDICTOR_VARIABLES_MEAS1,
        predictors_meas2=train_model_delta_unburned.PREDICTOR_VARIABLES_MEAS2,
        output_variable=train_model_delta_unburned.OUTPUT_VARIABLE,
    )

    fia_data_train = fia_data.where(test_train_split["is_train"], drop=True)
    fia_data_train_burned, fia_data_train_undisturbed = train_all_models.split_subcomponents(
        fia_data_train
    )

    [X_train_init, y_train_init] = train_models_utils.get_X_y(
        fia_data=fia_data_train,
        fia_variable_predictors=train_model_init_biomass.FIA_VARIABLE_PREDICTORS,
        predictors_meas1=train_model_init_biomass.PREDICTOR_VARIABLES_MEAS1,
        predictors_meas2=train_model_init_biomass.PREDICTOR_VARIABLES_MEAS2,
        output_variable=train_model_init_biomass.OUTPUT_VARIABLE,
    )
    [X_train_burned, y_test_burned] = train_models_utils.get_X_y(
        fia_data=fia_data_train_burned,
        fia_variable_predictors=train_model_delta_burned.FIA_VARIABLE_PREDICTORS,
        predictors_meas1=train_model_delta_burned.PREDICTOR_VARIABLES_MEAS1,
        predictors_meas2=train_model_delta_burned.PREDICTOR_VARIABLES_MEAS2,
        output_variable=train_model_delta_burned.OUTPUT_VARIABLE,
    )
    [X_train_undisturbed, y_train_undisturbed] = train_models_utils.get_X_y(
        fia_data=fia_data_train_undisturbed,
        fia_variable_predictors=train_model_delta_unburned.FIA_VARIABLE_PREDICTORS,
        predictors_meas1=train_model_delta_unburned.PREDICTOR_VARIABLES_MEAS1,
        predictors_meas2=train_model_delta_unburned.PREDICTOR_VARIABLES_MEAS2,
        output_variable=train_model_delta_unburned.OUTPUT_VARIABLE,
    )

    y_pred_init_test = model_initial_biomass.predict(X_test_init)
    y_pred_burned_test = model_burned.predict(X_test_burned)
    y_pred_undisturbed_test = model_undisturbed.predict(X_test_undisturbed)

    y_test_init_actual_list.append(y_test_init.values)
    y_test_burned_actual_list.append(y_test_burned.values)
    y_test_undisturbed_actual_list.append(y_test_undisturbed.values)
    y_test_init_pred_list.append(y_pred_init_test)
    y_test_burned_pred_list.append(y_pred_burned_test)
    y_test_undisturbed_pred_list.append(y_pred_undisturbed_test)

    plotid_init_list.append(y_test_init.index)
    plotid_burned_list.append(y_test_burned.index)
    plotid_undisturbed_list.append(y_test_undisturbed.index)

/dfs10/jranders_lab1/users/czarakas/uncertain_land_sink_data/processed_data/models/train_plotids_0000.nc


/opt/apps/python/3.10.2/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/apps/python/3.10.2/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
y_test_init = np.concatenate(y_test_init_actual_list)
y_test_burned = np.concatenate(y_test_burned_actual_list)
y_test_undisturbed = np.concatenate(y_test_undisturbed_actual_list)
y_pred_init_test = np.concatenate(y_test_init_pred_list)
y_pred_burned_test = np.concatenate(y_test_burned_pred_list)
y_pred_undisturbed_test = np.concatenate(y_test_undisturbed_pred_list)

In [7]:
columns_rename_dict = {
    "Ecosection_code": "Ecosection",
    "Ecoprovince_code": "Ecoprovince",
    "SLOPE_preliminary": "Slope",
    "ASPECT_preliminary": "Aspect",
    "ELEV_preliminary": "Elevation",
    "lat": "Latitude",
    "lon": "Longitude",
    "pct_own_public": "% Public ownership",
    "LIVE_CANOPY_CVR_PCT_start": "% Live canopy cover",
    "years_after_fire_start": "Years after fire",
    "STDAGE_start": "Stand age",
    "delta_live_canopy_cvr_pct": "% Live canopy cover change",
    "delta_live_canopy_cvr_pct_per_year": "% Live canopy cover change / year",
    "years_after_fire": "Years after fire",
    "biomass_start": "Initial biomass",
    "ppt_clim_maxseason": "Wet season mean precipitation",
    "ppt_clim_minseason": "Dry season mean precipitation",
    "ppt_clim_mean": "Mean precipitation",
    "tmean_clim_maxseason": "Hottest season mean temperature",
    "tmean_clim_minseason": "Coldest season mean temperature",
    "tmean_clim_mean": "Mean temperature",
    "tmin_clim_maxseason": "Hottest season min temperature",
    "tmin_clim_minseason": "Coldest season min temperature",
    "tmin_clim_mean": "Mean min temperature",
    "tmax_clim_maxseason": "Hottest season max temperature",
    "tmax_clim_minseason": "Coldest season max temperature",
    "tmax_clim_mean": "Mean max temperature",
    "vpdmax_clim_maxseason": "Driest season max VPD",
    "vpdmax_clim_minseason": "Humid season max VPD",
    "vpdmax_clim_mean": "Mean max VPD",
    "vpdmin_clim_maxseason": "Driest season min VPD",
    "vpdmin_clim_minseason": "Humid season min VPD",
    "vpdmin_clim_mean": "Mean min VPD",
}

In [8]:
def get_shap_values(X_train, X_test, model, do_subset=True):
    if do_subset:
        idx = np.random.choice(len(X_test), size=500, replace=False)
        X_test_sample = X_test.iloc[idx]
        explainer = shap.TreeExplainer(model, X_train)
        shap_values = explainer(X_test_sample, check_additivity=False)
    else:
        X_test_sample = X_test
        explainer = shap.TreeExplainer(model, X_train)
        shap_values = explainer(X_test_sample, check_additivity=False)

    X_test_sample = X_test_sample.rename(columns=columns_rename_dict)
    shap_values.feature_names = list(X_test_sample.columns)

    return X_test_sample, shap_values

In [9]:
X_test_sample_init, shap_values_init = get_shap_values(
    X_train=X_train_init, X_test=X_test_init, model=model_initial_biomass, do_subset=False
)

100%|===================| 7725/7729 [24:57<00:00]        

In [10]:
X_test_sample_burned, shap_values_burned = get_shap_values(
    X_train=X_train_burned, X_test=X_test_burned, model=model_burned, do_subset=False
)

 99%|===================| 547/553 [01:01<00:00]        

In [11]:
X_test_sample_undisturbed, shap_values_undisturbed = get_shap_values(
    X_train=X_train_undisturbed, X_test=X_test_undisturbed, model=model_undisturbed, do_subset=False
)

100%|===================| 7299/7300 [36:55<00:00]        

In [12]:
# Save SHAP Explanation objects
joblib.dump(shap_values_init, OUT_DIR + "shap_values_init.pkl")
joblib.dump(shap_values_burned, OUT_DIR + "shap_values_burned.pkl")
joblib.dump(shap_values_undisturbed, OUT_DIR + "shap_values_undisturbed.pkl")

# Save X_test_sample DataFrames (needed for column-index look-ups in plotting notebook)
X_test_sample_init.to_pickle(OUT_DIR + "X_test_sample_init.pkl")
X_test_sample_burned.to_pickle(OUT_DIR + "X_test_sample_burned.pkl")
X_test_sample_undisturbed.to_pickle(OUT_DIR + "X_test_sample_undisturbed.pkl")

print("Saved all SHAP artefacts to", OUT_DIR)

Saved all SHAP artefacts to ../figure_data/figure_S_SHAP/
